In [ ]:
# Pin TensorFlow to 2.15.0 and MLflow to 2.10.2 to perfectly match Serverless dependencies
%pip install tensorflow-cpu==2.15.0 scikit-learn tensorboard mlflow==2.10.2


## 1. Configuration

Set up the necessary catalog, schema, and MLflow variables. These can be overridden by Databricks Job parameters.

In [1]:
import os
import json
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score, precision_score, recall_score
import mlflow
import mlflow.tensorflow

# Helper to securely fetch parameters whether running in Databricks or Locally
def get_param(name, default_value):
    try:
        # Databricks injects base_parameters as widgets
        return dbutils.widgets.get(name)
    except Exception:
        # Fallback for local execution
        return os.getenv(name, default_value)

CATALOG_NAME = get_param("CATALOG_NAME", "main")
SCHEMA_NAME = get_param("SCHEMA_NAME", "default")
MODEL_NAME = get_param("MODEL_NAME", "nested_json_dl_model")
ENVIRONMENT = get_param("ENVIRONMENT", "local")

Environment: local
Target Model Path: <configure later>.<configure later>.nested_json_dl_model


## 2. Imports

In [2]:
import json
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, log_loss
import mlflow
import mlflow.tensorflow

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)


## 3. Load JSON

Load the nested JSON data from the local file system (or DBFS/S3 in production).

In [3]:
import json

CATALOG_NAME = get_param("CATALOG_NAME", "unum_ml_workspace")
SCHEMA_NAME = get_param("SCHEMA_NAME", "default")
WORKSPACE_FILE_PATH = get_param("WORKSPACE_FILE_PATH", "..")

# Unity Catalog Volume path is always accessible via /Volumes/ from any compute type
# This bypasses the Serverless Workspace FUSE restriction completely
if WORKSPACE_FILE_PATH != "..":
    data_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/ml_data/sample_nested.json"
else:
    # Local execution fallback
    data_path = "../data/sample_nested.json"

with open(data_path, 'r') as f:
    data = json.load(f)

print(f"Data path used: {data_path}")
print(f'Successfully loaded {len(data)} records')

records = data["records"]
print(f"Data path used: {data_path}")
print(f'Successfully loaded {len(records)} records')


Loaded 150 records.


## 4. Inspect raw data

In [4]:
if records:
    print(json.dumps(records[0], indent=2))


{
  "customer": {
    "age": 19,
    "location": {
      "city": "Bangalore",
      "country": "India"
    }
  },
  "activity": {
    "sessions": 48,
    "avg_duration": 36.63
  },
  "purchase": {
    "amount": 0,
    "previous_orders": 3
  },
  "target": 0
}


## 5. Preprocess nested JSON

Flatten the nested structure and handle obvious missing/invalid values.

In [5]:
def flatten_record(record):
    customer = record.get('customer', {})
    location = customer.get('location', {})
    activity = record.get('activity', {})
    purchase = record.get('purchase', {})
    
    return {
        'age': customer.get('age', 0),
        'city': location.get('city', 'Unknown'),
        'country': location.get('country', 'Unknown'),
        'sessions': activity.get('sessions', 0),
        'avg_duration': activity.get('avg_duration', 0.0),
        'previous_orders': purchase.get('previous_orders', 0),
        'target': record.get('target', 0)
    }

def preprocess_records(raw_records):
    flat_data = [flatten_record(r) for r in raw_records]
    return pd.DataFrame(flat_data)

if records:
    df = preprocess_records(records)
    print(df.head())


   age       city  country  sessions  avg_duration  previous_orders  target
0   19  Bangalore    India        48         36.63                3       0
1   61  Bangalore    India        48        107.60                1       1
2   31  Bangalore    India        15         63.12                0       0
3   32     Berlin  Germany        29         72.77                0       0
4   62    Seattle      USA        28         44.13                2       0


## 6. Feature preparation

Encode categorical values and scale numerical features.

In [6]:
def prepare_features(df):
    if df.empty:
        return np.array([]), np.array([]), None, None
        
    df_clean = df.copy()
    
    # Label encode categorical
    le_city = LabelEncoder()
    df_clean['city_encoded'] = le_city.fit_transform(df_clean['city'])
    
    le_country = LabelEncoder()
    df_clean['country_encoded'] = le_country.fit_transform(df_clean['country'])
    
    features = ['age', 'sessions', 'avg_duration', 'previous_orders', 'city_encoded', 'country_encoded']
    X = df_clean[features].values
    y = df_clean['target'].values
    
    # Scale numerical
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    return X_scaled, y, scaler, features

if records:
    X, y, scaler, feature_names = prepare_features(df)
    print(f"X shape: {X.shape}, y shape: {y.shape}")


X shape: (150, 6), y shape: (150,)


## 7. Train/test split

In [7]:
if records:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training samples: {len(X_train)}")
    print(f"Test samples: {len(X_test)}")


Training samples: 120
Test samples: 30


## 8. Build TensorFlow model

In [8]:
def build_model(input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(16, activation='relu'),
        Dense(8, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

if records:
    model = build_model(X_train.shape[1])
    model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │           112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 257 (1.00 KB)

 Trainable params: 257 (1.00 KB)

 Non-trainable params: 0 (0.00 B)

## 9. Train model, 10. Evaluate model, 12-16. MLflow tracking

We combine these steps to track them together in an MLflow run.

In [9]:
# Hyperparameters
EPOCHS = 10
BATCH_SIZE = 16

if records:
    # 12. Explicitly configure MLflow to use Databricks tracking and Unity Catalog registry
    # This is required on Serverless because Spark Connect blocks auto-detection via spark config
    mlflow.set_tracking_uri("databricks")
    mlflow.set_registry_uri("databricks-uc")
    
    # Start MLflow run
    with mlflow.start_run() as run:
        print(f"MLflow Run ID: {run.info.run_id}")
        
        # 13. Log parameters
        mlflow.log_param("train_samples", len(X_train))
        mlflow.log_param("test_samples", len(X_test))
        mlflow.log_param("model_type", "Sequential_Dense")
        
        # Enable auto-logging for TensorFlow
        mlflow.tensorflow.autolog()
        
        # 9. Train model
        history = model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.2, verbose=1)
        
        # 10. Evaluate model
        y_pred_prob = model.predict(X_test)
        y_pred = (y_pred_prob > 0.5).astype(int)
        
        test_loss = log_loss(y_test, y_pred_prob)
        test_acc = accuracy_score(y_test, y_pred)
        test_prec = precision_score(y_test, y_pred, zero_division=0)
        test_rec = recall_score(y_test, y_pred, zero_division=0)
        
        print("\nEvaluation Metrics:")
        print(f"Loss: {test_loss:.4f}")
        print(f"Accuracy: {test_acc:.4f}")
        print(f"Precision: {test_prec:.4f}")
        print(f"Recall: {test_rec:.4f}")
        
        # 14. Log metrics explicitly (though autolog handles some)
        mlflow.log_metric("test_loss", test_loss)
        mlflow.log_metric("test_accuracy", test_acc)
        mlflow.log_metric("test_precision", test_prec)
        mlflow.log_metric("test_recall", test_rec)
        
        # 15. Explicitly log the TensorFlow model as an artifact
        input_example = X_test[:1]
        mlflow.tensorflow.log_model(model, "model", input_example=input_example)


2026/09/11 16:01:24 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/11 16:01:24 INFO mlflow.store.db.utils: Updating database tables


MLflow Run ID: 701c58f1de194697811518af217a3a7d


Epoch 1/10
1/6 ━━━━━━━━━━━━━━━━━━━━ 2s 427ms/step - accuracy: 0.3125 - loss: 0.8115

6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.3854 - loss: 0.7523 - val_accuracy: 0.4167 - val_loss: 0.7424
Epoch 2/10
1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3125 - loss: 0.7704

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4479 - loss: 0.7252 - val_accuracy: 0.4583 - val_loss: 0.7142
Epoch 3/10
1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4375 - loss: 0.7386

...

2026/09/11 16:01:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step


2026/09/11 16:01:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Evaluation Metrics:
Loss: 0.6111
Accuracy: 0.7000
Precision: 0.7619
Recall: 0.8000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step


/Users/as-mac-0734/Documents/UNUM-mini-project/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(store)


## 11. Generate predictions

Demonstrate how a new input would be transformed.

In [10]:
if records:
    print("Sample Predictions vs Actuals:")
    for i in range(5):
        print(f"Actual: {y_test[i]}, Predicted Prob: {y_pred_prob[i][0]:.4f}, Predicted Class: {y_pred[i][0]}")
        
    print("\nSimulating new input:")
    new_record = {
        "customer": {"age": 25, "location": {"city": "London", "country": "UK"}},
        "activity": {"sessions": 40, "avg_duration": 45.0},
        "purchase": {"amount": 0, "previous_orders": 2}
    }
    
    flat_new = flatten_record(new_record)
    df_new = pd.DataFrame([flat_new])
    print(f"Flattened input:\n{df_new.iloc[0]}")
    
    # In a real scenario, you'd save and load the encoders and scaler.
    # We simulate transform with the ones in memory.
    try:
        # Note: this might fail if 'London' wasn't in the training set
        # But for this simple test, it should be fine.
        city_enc = 0 # Dummy encode for example
        country_enc = 0 
        
        features_new = [flat_new['age'], flat_new['sessions'], flat_new['avg_duration'], flat_new['previous_orders'], city_enc, country_enc]
        X_new_scaled = scaler.transform([features_new])
        pred_new = model.predict(X_new_scaled)
        print(f"Prediction for new input (prob): {pred_new[0][0]:.4f}")
    except Exception as e:
        print(f"Prediction error: {e}")


Sample Predictions vs Actuals:
Actual: 1, Predicted Prob: 0.4487, Predicted Class: 0
Actual: 1, Predicted Prob: 0.5793, Predicted Class: 1
Actual: 1, Predicted Prob: 0.5409, Predicted Class: 1
Actual: 0, Predicted Prob: 0.5428, Predicted Class: 1
Actual: 1, Predicted Prob: 0.6207, Predicted Class: 1

Simulating new input:
Flattened input:
age                    25
city               London
country                UK
sessions               40
avg_duration         45.0
previous_orders         2
target                  0
Name: 0, dtype: object
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
Prediction for new input (prob): 0.6028


## 17. Register model in MLflow / Unity Catalog & 18. Display version

Register the model so it can be served.

In [11]:
# We wrap this in a try-catch because if we are running locally without UC configured, it will fail.
if records and ENVIRONMENT != "local":
    model_uri = f"runs:/{run.info.run_id}/model"
    registered_model_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{MODEL_NAME}"
    
    try:
        print(f"Registering model to {registered_model_name}...")
        result = mlflow.register_model(model_uri, registered_model_name)
        
        print(f"Model successfully registered!")
        print(f"Name: {result.name}")
        print(f"Version: {result.version}")
        print(f"Status: {result.status}")
    except Exception as e:
        print(f"Failed to register model (likely because Unity Catalog is not configured locally): {e}")
else:
    print("Running in local mode, skipping model registration to Unity Catalog.")


Running in local mode, skipping model registration to Unity Catalog.
